In [ ]:
!pip3 install openai

In [ ]:
from openai import OpenAI
import json
import os

# -------- CONFIG (GROQ ONLY) --------
client = OpenAI(
    api_key="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",
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "openai/gpt-oss-120b" 

# -------- USER INPUT --------
userRequestData = {
    "level": "middle_school",
    "subject": "math",
    "topics": "solving linear equation, fraction",
    "numModules": "1",
    "moduleTypes": { "1": "mcq_4" },
    "moduleQuestionCounts": { "1": 5 },
    "multipleCorrect": "no",
    "explanations": "yes",
}

num_questions = userRequestData["moduleQuestionCounts"]["1"]

# -------- PROMPT BUILDER --------
def build_prompt():
    return f"""
You are an AI that generates structured educational assignments.

You MUST return ONLY valid JSON. No explanations, no markdown, no extra text.

Follow this EXACT schema:

{{
  "questions": {{
    "1": {{
      "question": "string",
      "options": ["string", "string", "string", "string"],
      "questionType": "single",
      "points": 0
    }}
  }},
  "correctAnswers": [[number]],
  "explanations": ["string"]
}}

Rules:
- Generate exactly {num_questions} questions
- Each question must have exactly 4 options
- Only ONE correct answer per question
- correctAnswers must contain the index (0-based)
- The correct answer MUST match the correct option
- explanations must correctly justify the answer
- explanations must align with the correct answer
- points MUST always be 0
- Output must be STRICT JSON
- No text outside JSON
- Double-check correctness before returning

Now generate:

Level: {userRequestData["level"]}
Subject: {userRequestData["subject"]}
Topics: {userRequestData["topics"]}
"""

# -------- STRUCTURE VALIDATION --------
def validate_output(data):
    try:
        questions = data["questions"]
        answers = data["correctAnswers"]
        explanations = data["explanations"]

        if not (len(questions) == len(answers) == len(explanations) == num_questions):
            return False

        for i, (q_id, q_data) in enumerate(questions.items()):
            options = q_data["options"]
            correct_index = answers[i][0]

            if len(options) != 4:
                return False
            if not (0 <= correct_index < 4):
                return False
            if len(explanations[i].strip()) < 10:
                return False
            if q_data["points"] != 0:
                return False

        return True

    except:
        return False

# -------- SELF-VERIFICATION (GROQ) --------
def verify_with_model(data):
    verify_prompt = f"""
You are verifying a multiple choice assignment.

For each question:
- Check if the marked correct answer is actually correct
- Check if the explanation matches the correct answer

Return ONLY JSON:

{{
  "valid": true or false
}}

Assignment:
{json.dumps(data)}
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": verify_prompt}],
        temperature=0
    )

    try:
        result = json.loads(response.choices[0].message.content)
        return result.get("valid", False)
    except:
        return False

# -------- GENERATION WITH RETRY --------
def generate_assignment(max_retries=5):
    for attempt in range(max_retries):
        print(f"Attempt {attempt + 1}...")

        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": build_prompt()}],
            temperature=0.2
        )

        output = response.choices[0].message.content

        try:
            parsed = json.loads(output)

            if not validate_output(parsed):
                print("Structure invalid, retrying...")
                continue

            if not verify_with_model(parsed):
                print("Logic invalid, retrying...")
                continue

            print("Fully valid output")
            return parsed

        except:
            print("JSON parsing failed, retrying...")

    raise Exception("Failed after retries")

# -------- RUN --------
result = generate_assignment()
print(json.dumps(result, indent=2))